##### Задача 5*

У меня мало опыта в написании запросов на диалекте Clichouse, но я могу написать на TSQL и перевести на любой другой диалект, одновременно подтгивая особенности другого диалекта. Данную задачу решил именно по такому методу

In [ ]:
-- объявляем переменные для указания временных границ вывода отчета (на 09-01 данные включаются по 25.08)
DECLARE @date_from date = '2026-09-01';   
DECLARE @date_to   date = '2026-09-30';   

WITH daily AS (
    -- действие пользователей в указанный диапазон -6 дней в начале
    SELECT DISTINCT
        CAST([timestamp] AS date) AS activity_date,
        user_id
    FROM dwh.site_activity
    WHERE [timestamp] >= DATEADD(day, -6, @date_from)
      AND [timestamp] <  DATEADD(day,  1, @date_to)
),
calendar AS (
    -- даты за которые считаем целевой показатель WAU (создаем таблицу с датами)
    SELECT @date_from AS dt
    UNION ALL
    SELECT DATEADD(day, 1, dt)
    FROM calendar
    WHERE dt < @date_to
)

SELECT
    c.dt AS [date],
    COUNT(DISTINCT d.user_id) AS wau -- кол-во уникальных активностей 
FROM calendar AS c
LEFT JOIN daily AS d
       ON d.activity_date BETWEEN DATEADD(day, -6, c.dt) AND c.dt -- связка не точная, а по диапазону 


GROUP BY c.dt
ORDER BY c.dt
OPTION (MAXRECURSION 0)

Попросим переписать ИИ код который был написан на другом диалекте

In [ ]:
SELECT
    day                 AS date,
    uniqExact(user_id)  AS wau
FROM
(
    SELECT DISTINCT
        toDate(timestamp) AS activity_date,
        user_id
    FROM dwh.site_activity
    WHERE toDate(timestamp) >= toDate('2026-09-01') - 6 
      AND toDate(timestamp) <= toDate('2026-09-30')
)
ARRAY JOIN arrayMap(i -> activity_date + i, range(7)) AS day

WHERE day >= toDate('2026-09-01')
  AND day <= toDate('2026-09-30')

GROUP BY day
ORDER BY day

После перевода в другой диалект изучил весь код,

ARRAY JOIN arrayMap(i -> activity_date + i, range(7)) AS day

основная разница заключается в использовании конструкции выше, т.к. кликхаус плохо работает с джойнами по диапазону поэтому здесь применяется обратная логика. Каждая активность пользователей раздается на 7 дней вперед. Т.е. включается в показатель WAU на 7 дней вперед.